# TOS<sup>2</sup>CA Data Curation End-To-End Example

This notebook is meant to be an example of how to run a Data Curatyion job in an end-to-end fashion.  It reads/subsets data in time and space, interploates the data to a common grid (i.e., the same grid used to from the input dataset to generate the masks in PhDef), and stitches the chunks together into a single file.  It does not include any steps in the Phenomenon Definition (PhDef) stage of TOS<sup>2</sup>CA.

## Import Python Libraries

First, we'll import the necessary Python libraries.  This assumes you already have [tos2ca-anomaly-detection](https://github.com/nasa-jpl/tos2ca-anomaly-detection) and [tos2ca-fortracc-module](https://github.com/nasa-jpl/tos2ca-fortracc-module) installed and in your `PYTHONPATH`.  You should also have installed the [data dictionaries](https://github.com/nasa-jpl/tos2ca-data-dictionaries) on your system and updated any paths in the code to point to them.

In [ ]:
import sys
from iolib.gpm import gpm_curator
from utils.interpolation import interpolator
from utils.ncTools import combineCuratedFiles, combineInterpolatedFiles, cleanUpChunks

## Job Parameters

The job we're going to run has the following parameters, which you'll need to insert into your MySQL database instance's `jobs` table (see the [database architecture](../../db/tosca_db.sql) to get going on that if you haven't installed it already).  It also requires that an entry for jobID 390 (the PhDef job this Data Curation job is based on) exists in the same `jobs` table.

```mysql
jobID: 399
userID: 1
nChunks: 1
stage: curation
phdefJobID: 390
dataset: GPM_MERGIR
variable: Tb
coords: NULL
startDate: NULL
endDate: NULL
ineqOperator: NULL
ineqValue: NULL
description: NULL
status: pending
submitTime: 2024-07-23 20:42:07
```

The job (399) only has one chunk for simplicty here, but jobs can have <i>n</i> number of chunks.  This also assumes you've chunked your job into chunks and have an entry for chunkID 1 in your `chunks` table.  It also requires that you have a user in your `users` table with a userID of 1.

Note that this job happens to use GPM data, which is why we imported `gpm_curator` in the imports block.

## Setup Global Variables

Again, we're using jobID #390 and chunkID #1

In [ ]:
jobID = 399
chunkID = 1

## Read and Subset the Data

This will output a netCDF-4 file and JSON hierarchy file to your S3 bucket for each chunk.

In [ ]:
print("Running jobID: %s-%s" % (jobID, chunkID))
gpm_curator(jobID, chunkID)
print("Finished curating chunk.")

## Stitch Curated Files Together

This will output a single netCDF-4 curated data file and JSON hierarchy file to your S3 bucket.  This can take a while, and isn't strictly necessary if all you need is the interpolated files.

In [ ]:
print("Combine chunks into single file")
combineCuratedFiles(jobID)
print("Finished stitching curated file")

## Interpolate the Data

This will output a netCDF-4 file and JSON architecture file to your S3 bucket for each chunk.

In [ ]:
print("Run Interpolater")
interpolator(jobID, chunkID)
print("Done interpolating")

## Stitch Interpolated Files Together 

This will output a single netCDF-4 curated data file and JSON hierarchy file to your S3 bucket.

In [ ]:
print("Combine chunks into single file")
combineInterpolatedFiles(jobID)
print("Finished stitching interpolated file")

## Cleanup

Now that you've generated stitched file, you can cleaup the chunked files, leaving only the stitched files.

In [ ]:
print("Removing chunked files")
cleanUpChunks(jobID)
print("Finished removing chunked files")

## Outputs

All outputs will output to the S3 bucket name that you have identified in AWS secrets (see what [secrets you need to setup](../../templates/aws_secrets.json)):

```
s3://<bucket name>/<jobID>/
```